In [ ]:
import pandas as pd
from pathlib import Path
import os
import openpyxl
import re
import json
import pandas as pd

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SAMPLES =  PROJECT_ROOT / "data" / "samples"
BENCHMARKS_FINAL = PROJECT_ROOT / "benchmarks" / "final"

In [ ]:
deepseek = pd.read_csv(BENCHMARKS_FINAL / "deepseek_deepseek-r1-0528-qwen3-8b_n200_maxtok2000_20260407_105237.csv")[["transcript_id", "output"]]
qwen = pd.read_csv(BENCHMARKS_FINAL / "qwen_qwen3-4b-2507_n200_maxtok2000_20260407_103222.csv")[["transcript_id", "output"]]

In [ ]:
deepseek = deepseek.add_suffix("_deepseek")
qwen     = qwen.add_suffix("_qwen")

# restore key column name
for df in [deepseek, qwen]:
    df.rename(columns={df.columns[0]: "transcript_id"}, inplace=True)

In [ ]:
merge = (
    deepseek
    .merge(qwen, on="transcript_id")
)

In [ ]:
merge.columns

In [ ]:


def parse_llm_output(text):
    if pd.isna(text):
        return None
    
    # --- clean markdown ---
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```", "", text)
    
    # --- fix double quotes from CSV ---
    text = text.replace('""', '"')
    
    # --- extract balanced JSON blocks ---
    stack = []
    json_blocks = []
    start = None

    for i, char in enumerate(text):
        if char == "{":
            if not stack:
                start = i
            stack.append(char)
        elif char == "}":
            if stack:
                stack.pop()
                if not stack and start is not None:
                    json_blocks.append(text[start:i+1])

    # --- parse JSON ---
    parsed = []
    for block in json_blocks:
        try:
            parsed.append(json.loads(block))
        except:
            pass

    return parsed if parsed else None

In [ ]:
merge["deepseek_parsed"] = merge["output_deepseek"].apply(parse_llm_output)
merge["qwen_parsed"]     = merge["output_qwen"].apply(parse_llm_output)

In [ ]:
def parsed_to_long(df, parsed_col, source_name):
    rows = []

    for _, row in df.iterrows():
        tid = row["transcript_id"]
        parsed_list = row[parsed_col]

        if not parsed_list:
            continue

        for d in parsed_list:
            for k, v in d.items():

                # --- handle nested context_summary ---
                if isinstance(v, dict):
                    for subk, subv in v.items():
                        rows.append({
                            "transcript_id": tid,
                            "Measure": subk,
                            "Score": subv,
                            "Source": source_name
                        })
                else:
                    rows.append({
                        "transcript_id": tid,
                        "Measure": k,
                        "Score": v,
                        "Source": source_name
                    })

    return pd.DataFrame(rows)

In [ ]:
deepseek_long = parsed_to_long(merge, "deepseek_parsed", "deepseek")
qwen_long     = parsed_to_long(merge, "qwen_parsed", "qwen")

In [ ]:
final_long = pd.concat([
    deepseek_long,
    qwen_long,
])

In [ ]:
final_long["Measure"].unique()

In [ ]:
numeric_measures = ["forward_looking_intensity", "specificity", "economic_substance", "tone", "certainty"]

df_num = final_long[final_long["Measure"].isin(numeric_measures)].copy()

df_num["Score"] = pd.to_numeric(df_num["Score"], errors="coerce")

pivot = df_num.pivot_table(
    index=["transcript_id", "Measure"],
    columns="Source",
    values="Score"
)

pivot = pivot.dropna()

pivot.corr()

In [ ]:
pivot.std()

In [ ]:
pivot.mean()

In [ ]:
results = []

for measure, group in pivot.reset_index().groupby("Measure"):
    corr = group[["deepseek", "qwen"]].corr().loc["deepseek"]
    corr["Measure"] = measure
    results.append(corr)

corr_table = pd.DataFrame(results).set_index("Measure")
print(corr_table)